# OCR BNP — Pipeline InternVL3-8B + JSON checkpoint
> `Kernel → Restart Kernel and Run All Cells`


## 1. Dépendances

In [ ]:
%pip install -q pymupdf pillow transformers openpyxl torchvision einops
print('✅ OK')

## 2. Imports

In [ ]:
import time, json, re, gc
import numpy as np
from pathlib import Path
from datetime import datetime
from collections import defaultdict
import fitz
import torch
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
from PIL import Image
from transformers import AutoTokenizer, AutoModel
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
print('✅ Imports OK')

## 3. Config

In [ ]:
MODEL_PATH      = "/domino/edv/modelhub/ModelHub-model-huggingface-OpenGVLab/InternVL3-8B/main"
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
MAX_NEW_TOKENS  = 3000
IMAGE_MAX_SIZE  = 2024
PDF_ZOOM        = 3.0
BLANK_THRESHOLD = 0.95
GPU_BATCH_SIZE  = 4

INPUT_DIR  = Path("/mnt/data/transferts_in/2025_11")
OUTPUT_DIR = Path("/mnt/data/transferts_out")
JSON_DIR   = OUTPUT_DIR / "json_dossiers"
LOG_PATH   = OUTPUT_DIR / "pipeline.log"
EXCEL_PATH = OUTPUT_DIR / f"audit_transferts_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JSON_DIR.mkdir(parents=True, exist_ok=True)

pdfs = sorted(INPUT_DIR.glob('*.pdf'))
print(f'Device         : {DEVICE}')
print(f'GPU batch size : {GPU_BATCH_SIZE}')
print(f'Dossiers       : {len(pdfs)}')
print(f'JSON dir       : {JSON_DIR}')
print(f'Excel          : {EXCEL_PATH}')

## 4. Chargement modèle InternVL3-8B

In [ ]:
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH, trust_remote_code=True
)
model = AutoModel.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
).eval().to(DEVICE)
print(f'✅ InternVL3-8B chargé en {time.time()-t0:.1f}s')
print(f'   Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'   VRAM   : {torch.cuda.memory_allocated()/1e9:.1f}GB utilisés')

## 5. Utilitaires PDF & inférence

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def build_transform(input_size=448):
    return T.Compose([
        T.Lambda(lambda img: img.convert('RGB')),
        T.Resize((input_size, input_size),
                  interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def resize(img, max_side=IMAGE_MAX_SIZE):
    w, h = img.size
    if max(w, h) <= max_side: return img
    r = max_side / max(w, h)
    return img.resize((int(w*r), int(h*r)), Image.LANCZOS)


def is_blank(image, threshold=BLANK_THRESHOLD) -> bool:
    arr = np.array(image.convert('L'))
    return (arr > 240).sum() / arr.size >= threshold


def pdf_to_pages(path: Path, zoom=PDF_ZOOM) -> list:
    doc    = fitz.open(path)
    matrix = fitz.Matrix(zoom, zoom)
    pages  = []
    for i in range(len(doc)):
        pix = doc.load_page(i).get_pixmap(matrix=matrix, alpha=False)
        img = resize(Image.frombytes('RGB', [pix.width, pix.height], pix.samples))
        pages.append({'index': i, 'image': img})
    doc.close()
    return pages


def parse_json(text: str) -> dict:
    if not text:
        return {}
    text = text.strip()
    text = text.replace('```json', '').replace('```', '').strip()
    try:
        m = re.search(r'[{].*[}]', text, re.S)
        return json.loads(m.group()) if m else {}
    except Exception:
        return {}


def ask_single(prompt: str, image) -> dict:
    """
    Inférence InternVL3 via model.chat().
    Retourne {text, tokens_in, tokens_out, elapsed}.
    """
    transform    = build_transform(input_size=448)
    pixel_values = transform(image).unsqueeze(0).to(torch.float16).to(DEVICE)
    question     = '<image>' + chr(10) + prompt

    generation_config = {
        'max_new_tokens': MAX_NEW_TOKENS,
        'do_sample':      False,
    }

    t0 = time.time()
    with torch.no_grad():
        response = model.chat(
            tokenizer        = tokenizer,
            pixel_values     = pixel_values,
            question         = question,
            generation_config= generation_config,
        )
    elapsed = time.time() - t0

    # InternVL3 peut retourner (text, history) ou juste text
    if isinstance(response, tuple):
        text = response[0]
    elif isinstance(response, str):
        text = response
    elif response is not None:
        text = str(response)
    else:
        text = ''

    tok_out = len(tokenizer.encode(text)) if text else 0

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        'text':       text,
        'tokens_in':  0,
        'tokens_out': tok_out,
        'elapsed':    round(elapsed, 2),
    }


def ask_batch(prompt: str, images: list) -> list:
    """
    Batch InternVL3 — traite N images.
    Fallback ask_single si OOM ou erreur.
    """
    if not images:
        return []
    # InternVL3 batch via ask_single sequentiel — plus stable
    results = []
    for img in images:
        try:
            results.append(ask_single(prompt, img))
        except Exception as e:
            print(f'  ⚠️  Erreur page : {e}')
            results.append({'text': '{}', 'tokens_in': 0,
                            'tokens_out': 0, 'elapsed': 0.0})
    return results


print('✅ Utilitaires InternVL3 OK')

## 6. Normalisation des données

In [ ]:
def norm_str(v) -> str:
    if v is None: return None
    s = re.sub(r'\s+', ' ', str(v).strip())
    return s if s and s.lower() not in ('null', 'none', 'n/a') else None

def norm_upper(v) -> str:
    s = norm_str(v)
    return s.upper() if s else None

def norm_compte(v) -> str:
    s = norm_str(v)
    if not s: return None
    return re.sub(r'[^A-Za-z0-9]', '', s).upper()

def norm_montant(v) -> float:
    if v is None: return None
    if isinstance(v, (int, float)): return float(v)
    s = str(v).strip()
    s = re.sub(r'[^\d.,]', '', s)
    if not s: return None
    if s.count(',') == 1 and '.' not in s: s = s.replace(',', '.')
    elif '.' in s and ',' in s:            s = s.replace(',', '')
    elif s.count(',') > 1:                 s = s.replace(',', '')
    try: return float(s)
    except: return None

def norm_date(v) -> str:
    if not v: return None
    s = norm_str(v)
    if not s: return None
    if re.match(r'^\d{2}/\d{2}/\d{4}$', s): return s
    m = re.match(r'^(\d{4})-(\d{2})-(\d{2})$', s)
    if m: return f'{m.group(3)}/{m.group(2)}/{m.group(1)}'
    return s

def norm_periode(v) -> str:
    if not v: return None
    s = str(v).upper().strip()
    s = re.sub(r'[._\-]', ' ', s)
    s = re.sub(r'PART\s*(\d)', r'PART \1', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

CHAMPS_OV   = {'type','monnaie','montant_chiffres','montant_lettres',
               'periode','tranche','date_demande','compte_debiteur',
               'beneficiaire_nom','beneficiaire_compte','beneficiaire_adresse',
               'swift','banque_nom'}
CHAMPS_ANN1 = {'type','nom_prenom_client','compte_bancaire',
               'nom_prenom_signataire','date_signature'}
CHAMPS_ANN2 = {'type','mois_transfert','nom_prenom','compte_bancaire',
               'salaire_mensuel','nombre_jours','part_transferable',
               'pays_destination','nom_banque','numero_compte_etranger',
               'numero_domiciliation'}
CHAMPS_BP   = {'type','nom_prenom_salarie','matricule','mois_bulletin',
               'salaire_base','salaire_brut','retenue_ss','retenue_irg',
               'retenue_mutuelle','net_a_payer'}

def normalise_doc(doc_type: str, data: dict) -> dict:
    d = dict(data)
    if doc_type == 'OV':
        for key in list(d.keys()):
            if isinstance(d.get(key), dict):
                d.update(d.pop(key))
        if not d.get('compte_debiteur'):
            for alias in ['compte_debiteur_donneur_ordre','compte_donneur',
                          'numero_compte','compte_debiteur_ordre']:
                if d.get(alias):
                    d['compte_debiteur'] = d[alias]
                    break
        if d.get('tranche') in ('70','32','50','59','57',70,32,50,59,57):
            d['tranche'] = None
        if d.get('periode') and re.match(r'^\d{2}/\d{2}/\d{4}$',
                                          str(d.get('periode',''))):
            d['periode'] = None
        d = {k: v for k, v in d.items() if k in CHAMPS_OV}
        d['monnaie']              = norm_upper(d.get('monnaie'))
        d['montant_chiffres']     = norm_montant(d.get('montant_chiffres'))
        d['montant_lettres']      = norm_str(d.get('montant_lettres'))
        d['periode']              = norm_periode(d.get('periode'))
        d['tranche']              = norm_str(d.get('tranche'))
        d['date_demande']         = norm_date(d.get('date_demande'))
        d['compte_debiteur']      = norm_compte(d.get('compte_debiteur'))
        d['beneficiaire_nom']     = norm_upper(d.get('beneficiaire_nom'))
        d['beneficiaire_compte']  = norm_compte(d.get('beneficiaire_compte'))
        d['beneficiaire_adresse'] = norm_str(d.get('beneficiaire_adresse'))
        d['swift']                = norm_compte(d.get('swift'))
        d['banque_nom']           = norm_upper(d.get('banque_nom'))
    elif doc_type == 'ANNEXE_I':
        d = {k.lower(): v for k, v in d.items()}
        aliases = {
            'nom_prenom_employe': 'nom_prenom_client',
            'nom_prenom_employé': 'nom_prenom_client',
            'prenom_nom':         'nom_prenom_client',
            'nom':                'nom_prenom_client',
            'date_siagnature':    'date_signature',
            'date_de_signature':  'date_signature',
            'nom_signataire':     'nom_prenom_signataire',
        }
        for old, new in aliases.items():
            if old in d and new not in d:
                d[new] = d.pop(old)
        d = {k: v for k, v in d.items() if k in CHAMPS_ANN1}
        d['nom_prenom_client']     = norm_upper(d.get('nom_prenom_client'))
        d['compte_bancaire']       = norm_compte(d.get('compte_bancaire'))
        d['nom_prenom_signataire'] = norm_upper(d.get('nom_prenom_signataire'))
        d['date_signature']        = norm_date(d.get('date_signature'))
    elif doc_type == 'ANNEXE_II':
        d = {k: v for k, v in d.items() if k in CHAMPS_ANN2}
        d['mois_transfert']         = norm_periode(d.get('mois_transfert'))
        d['nom_prenom']             = norm_upper(d.get('nom_prenom'))
        d['compte_bancaire']        = norm_compte(d.get('compte_bancaire'))
        d['salaire_mensuel']        = norm_montant(d.get('salaire_mensuel'))
        d['nombre_jours']           = norm_str(d.get('nombre_jours'))
        d['part_transferable']      = norm_montant(d.get('part_transferable'))
        d['pays_destination']       = norm_upper(d.get('pays_destination'))
        d['nom_banque']             = norm_str(d.get('nom_banque'))
        d['numero_compte_etranger'] = norm_compte(d.get('numero_compte_etranger'))
        d['numero_domiciliation']   = norm_str(d.get('numero_domiciliation'))
    elif doc_type == 'BULLETIN':
        d = {k: v for k, v in d.items() if k in CHAMPS_BP}
        d['nom_prenom_salarie'] = norm_upper(d.get('nom_prenom_salarie'))
        d['matricule']          = norm_str(d.get('matricule'))
        d['mois_bulletin']      = norm_periode(d.get('mois_bulletin'))
        d['salaire_base']       = norm_montant(d.get('salaire_base'))
        d['salaire_brut']       = norm_montant(d.get('salaire_brut'))
        d['retenue_ss']         = norm_montant(d.get('retenue_ss'))
        d['retenue_irg']        = norm_montant(d.get('retenue_irg'))
        d['retenue_mutuelle']   = norm_montant(d.get('retenue_mutuelle'))
        d['net_a_payer']        = norm_montant(d.get('net_a_payer'))
    return d

print('✅ Normalisation OK')

## 7. Prompt universel

In [ ]:
PROMPT_UNIVERSEL = (
    'Lis ce document bancaire.' + chr(10) + chr(10) +
    'ÉTAPE 1 — Identifie le type en lisant le titre principal :' + chr(10) +
    '- OV        : titre "ORDRE DE VIREMENT A L\'ETRANGER"' + chr(10) +
    '- ANNEXE_I  : titre "Annexe I", contient "Transfert sur salaire perçus en Algérie"' + chr(10) +
    '- ANNEXE_II : titre "Annexe II", contient "Fiche de paie spéciale relative à un transfert du salaire"' + chr(10) +
    '- BULLETIN  : titre "BULLETIN DE PAIE"' + chr(10) +
    '- AUTRE     : tout autre document (page vide, tableau, cachet...)' + chr(10) + chr(10) +
    'ÉTAPE 2 — Selon le type identifié, extrais uniquement les champs listés.' + chr(10) +
    'Si le type est AUTRE, retourne uniquement {"type": "AUTRE"}.' + chr(10) + chr(10) +
    '--- Si OV ---' + chr(10) +
    'Zone 32 : monnaie, montant_chiffres, montant_lettres' + chr(10) +
    'Zone 70 : periode (mois + année bruts ex: JANVIER 2026 / null),' + chr(10) +
    '          tranche (Part 1 / Part 2 / null. Ne pas confondre avec le numéro 70)' + chr(10) +
    'Zone 50 : date_demande, compte_debiteur (20 chiffres commençant par 02700)' + chr(10) +
    'Zone 59 : beneficiaire_nom, beneficiaire_compte (IBAN sans espaces), beneficiaire_adresse' + chr(10) +
    'Zone 57 : swift, banque_nom' + chr(10) + chr(10) +
    '--- Si ANNEXE_I ---' + chr(10) +
    'Prends le temps de lire toute la lettre avant d\'extraire.' + chr(10) +
    'nom_prenom_client (juste après "Je soussigné"),' + chr(10) +
    'compte_bancaire (numéro 20 chiffres),' + chr(10) +
    'nom_prenom_signataire,' + chr(10) +
    'date_signature (après "Bethioua le", format jj/MM/aaaa)' + chr(10) + chr(10) +
    '--- Si ANNEXE_II ---' + chr(10) +
    'mois_transfert (valeur brute après "Mois de"),' + chr(10) +
    'nom_prenom, compte_bancaire (numéro 20 chiffres),' + chr(10) +
    'salaire_mensuel, nombre_jours (null si absent),' + chr(10) +
    'part_transferable (montant chiffres uniquement),' + chr(10) +
    'pays_destination,' + chr(10) +
    'nom_banque (première partie après "Compte devise :"),' + chr(10) +
    'numero_compte_etranger (deuxième partie après "Compte devise :", sans espaces),' + chr(10) +
    'numero_domiciliation (dans le tableau "DOMICILIATION IMPORT", 5 colonnes ex: 271901|2024.2|40|00136|DZD, null si absent)' + chr(10) + chr(10) +
    '--- Si BULLETIN ---' + chr(10) +
    'nom_prenom_salarie, matricule, mois_bulletin,' + chr(10) +
    'salaire_base, salaire_brut, retenue_ss, retenue_irg, retenue_mutuelle, net_a_payer' + chr(10) + chr(10) +
    'RÈGLES :' + chr(10) +
    '- Retourne UNIQUEMENT un JSON valide, sans texte avant ni après' + chr(10) +
    '- Commence toujours par {"type": "..."}' + chr(10) +
    '- Si un champ est absent ou illisible : null' + chr(10) +
    '- Pas de markdown, pas de backticks' + chr(10) +
    '- N\'invente aucun champ supplémentaire'
)

print(f'✅ Prompt : {len(PROMPT_UNIVERSEL)} caractères')

## 8. Traitement d'un PDF

In [ ]:
def process_pdf(pdf_path: Path, verbose=True) -> dict:
    TYPES_ATTENDUS = {'OV', 'ANNEXE_I', 'ANNEXE_II', 'BULLETIN'}
    pages    = pdf_to_pages(pdf_path)
    results  = {}
    doublons = []
    total_tok_in  = 0
    total_tok_out = 0
    t_dossier     = time.time()

    if verbose:
        print(f'\n📁 {pdf_path.name} — {len(pages)} page(s)')

    for page in pages:
        i   = page['index']
        img = page['image']

        if is_blank(img):
            if verbose: print(f'  Page {i+1} → ignorée (vide)')
            continue

        rep      = ask_single(PROMPT_UNIVERSEL, img)
        data     = parse_json(rep['text'])
        doc_type = data.get('type', 'INCONNU')

        total_tok_in  += rep['tokens_in']
        total_tok_out += rep['tokens_out']

        if doc_type in ('AUTRE', 'INCONNU'):
            if verbose: print(f'  Page {i+1} → {doc_type} — ignorée | {rep["elapsed"]}s')
            continue

        data = normalise_doc(doc_type, data)

        if verbose:
            print(f'  Page {i+1} → {doc_type:10s} | {rep["elapsed"]}s | tok={rep["tokens_out"]}')
            for k, v in data.items():
                if k != 'type' and v is not None:
                    print(f'    {k:25s}: {v}')

        if doc_type in results:
            doublons.append(doc_type)
            if verbose: print(f'    ⚠️  DOUBLON {doc_type}')
            continue

        results[doc_type] = data

    pages_trouvees   = sorted(results.keys())
    pages_manquantes = sorted(TYPES_ATTENDUS - set(results.keys()))

    return {
        'fichier':          pdf_path.name,
        'date_traitement':  datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'temps_total_s':    round(time.time() - t_dossier, 2),
        'tokens_in':        total_tok_in,
        'tokens_out':       total_tok_out,
        'tokens_total':     total_tok_in + total_tok_out,
        'pages_trouvees':   ', '.join(pages_trouvees),
        'pages_manquantes': ', '.join(pages_manquantes) if pages_manquantes else None,
        'anomalies':        'DOUBLON: ' + ', '.join(doublons) if doublons else None,
        **{t: results.get(t, {}) for t in TYPES_ATTENDUS},
    }

print('✅ process_pdf OK')

## 9. Export Excel

In [ ]:
SCHEMA = {
    'META': [
        ('fichier',          'Fichier'),
        ('date_traitement',  'Date traitement'),
        ('temps_total_s',    'Temps (s)'),
        ('tokens_total',     'Tokens total'),
        ('pages_trouvees',   'Pages trouvées'),
        ('pages_manquantes', 'Pages manquantes'),
        ('anomalies',        'Anomalies'),
    ],
    'OV': [
        ('monnaie',              'Monnaie'),
        ('montant_chiffres',     'Montant (chiffres)'),
        ('montant_lettres',      'Montant (lettres)'),
        ('periode',              'Période'),
        ('tranche',              'Tranche'),
        ('date_demande',         'Date demande'),
        ('compte_debiteur',      'Compte débiteur'),
        ('beneficiaire_nom',     'Bénéficiaire nom'),
        ('beneficiaire_compte',  'Bénéficiaire compte'),
        ('beneficiaire_adresse', 'Bénéficiaire adresse'),
        ('swift',                'SWIFT'),
        ('banque_nom',           'Banque bénéficiaire'),
    ],
    'ANNEXE_I': [
        ('nom_prenom_client',     'Nom Prénom client'),
        ('compte_bancaire',       'Compte bancaire'),
        ('nom_prenom_signataire', 'Nom Prénom signataire'),
        ('date_signature',        'Date signature'),
    ],
    'ANNEXE_II': [
        ('mois_transfert',         'Mois transfert'),
        ('nom_prenom',             'Nom Prénom'),
        ('compte_bancaire',        'Compte bancaire'),
        ('salaire_mensuel',        'Salaire mensuel'),
        ('nombre_jours',           'Nombre de jours'),
        ('part_transferable',      'Part transférable'),
        ('pays_destination',       'Pays destination'),
        ('nom_banque',             'Nom banque'),
        ('numero_compte_etranger', 'Compte étranger'),
        ('numero_domiciliation',   'N° Domiciliation'),
    ],
    'BULLETIN': [
        ('nom_prenom_salarie', 'Nom Prénom'),
        ('matricule',          'Matricule'),
        ('mois_bulletin',      'Mois bulletin'),
        ('salaire_base',       'Salaire base'),
        ('salaire_brut',       'Salaire brut'),
        ('retenue_ss',         'Retenue SS'),
        ('retenue_irg',        'Retenue IRG'),
        ('retenue_mutuelle',   'Retenue mutuelle'),
        ('net_a_payer',        'Net à payer'),
    ],
}
COLORS = {
    'META':      {'header': 'FF1F4E79', 'col': 'FFD6E4F0'},
    'OV':        {'header': 'FF833C00', 'col': 'FFFCE4D6'},
    'ANNEXE_I':  {'header': 'FF375623', 'col': 'FFE2EFDA'},
    'ANNEXE_II': {'header': 'FF203864', 'col': 'FFDAE3F3'},
    'BULLETIN':  {'header': 'FF3F3151', 'col': 'FFEDE7F6'},
}

def create_excel(path: Path, rows: list):
    wb = Workbook()
    ws = wb.active
    ws.title = 'Dossiers'
    all_cols = [(g, k, l) for g, cols in SCHEMA.items() for k, l in cols]
    col_idx  = 1
    group_map = defaultdict(list)
    for g, _, _ in all_cols:
        group_map[g].append(col_idx)
        col_idx += 1
    for g, cols in group_map.items():
        s, e = cols[0], cols[-1]
        if s < e:
            ws.merge_cells(start_row=1, start_column=s, end_row=1, end_column=e)
        c = ws.cell(row=1, column=s)
        c.value = g
        c.font  = Font(bold=True, color='FFFFFFFF', name='Arial', size=11)
        c.fill  = PatternFill('solid', start_color=COLORS[g]['header'])
        c.alignment = Alignment(horizontal='center', vertical='center')
    ws.row_dimensions[1].height = 22
    for i, (g, _, label) in enumerate(all_cols, start=1):
        c = ws.cell(row=2, column=i)
        c.value = label
        c.font  = Font(bold=True, name='Arial', size=9)
        c.fill  = PatternFill('solid', start_color=COLORS[g]['col'])
        c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        c.border = Border(bottom=Side(style='thin'), right=Side(style='hair'))
        ws.column_dimensions[get_column_letter(i)].width = 20
    ws.row_dimensions[2].height = 35
    ws.freeze_panes = ws.cell(row=3, column=len(SCHEMA['META']) + 1)
    for row_num, dossier in enumerate(rows, start=3):
        for col_idx, (g, k, _) in enumerate(all_cols, start=1):
            val = dossier.get(k) if g == 'META' else (dossier.get(g) or {}).get(k)
            c = ws.cell(row=row_num, column=col_idx)
            c.value = val
            c.font  = Font(name='Arial', size=9)
            c.fill  = PatternFill('solid', start_color=COLORS[g]['col'])
            if isinstance(val, float):
                c.number_format = '0.00'
            if g == 'META' and k in ('anomalies', 'pages_manquantes') and val:
                c.font = Font(name='Arial', size=9, bold=True, color='FFCC0000')
    wb.save(path)
    print(f'✅ Excel : {path} | {len(rows)} dossiers | {len(all_cols)} colonnes')

print('✅ Export Excel OK')

## 10. Log

In [ ]:
def log(msg: str):
    ligne = f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} — {msg}"
    print(ligne)
    with open(LOG_PATH, 'a', encoding='utf-8') as f:
        f.write(ligne + chr(10))

print('✅ Log OK')

## 11. Test sur 1 dossier

In [ ]:
# Tester sur 1 dossier avant de lancer tout le batch
pdf_test = pdfs[0]
print(f'Test : {pdf_test.name}')
result   = process_pdf(pdf_test, verbose=True)
print(f'\nDocs trouvés : {result["pages_trouvees"]}')
if result.get('pages_manquantes'): print(f'Manquants : {result["pages_manquantes"]}')
if result.get('anomalies'):        print(f'Anomalies : {result["anomalies"]}')

## 12. Pipeline complet

In [ ]:
import psutil

deja_traites = {f.stem for f in JSON_DIR.glob('*.json')}
a_traiter    = [p for p in pdfs if p.stem not in deja_traites]
total_pdfs   = len(pdfs)

log(f'Total PDFs     : {total_pdfs}')
log(f'Déjà traités   : {len(deja_traites)}')
log(f'À traiter      : {len(a_traiter)}')
log(f'RAM libre      : {psutil.virtual_memory().available/1e9:.1f} GB')
if torch.cuda.is_available():
    log(f'VRAM libre     : {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

t_total       = time.time()
grand_tok_in  = 0
grand_tok_out = 0
n_ok = n_err  = 0

for num, pdf_path in enumerate(pdfs, start=1):
    if pdf_path.stem in deja_traites:
        log(f'[{num:>4}/{total_pdfs}] ⏭️  SKIP : {pdf_path.name}')
        continue
    try:
        result = process_pdf(pdf_path, verbose=False)
        json_file = JSON_DIR / f'{pdf_path.stem}.json'
        with open(json_file, 'w', encoding='utf-8') as f:
            json.dump(result, f, ensure_ascii=False, indent=2, default=str)
        grand_tok_in  += result.get('tokens_in',  0)
        grand_tok_out += result.get('tokens_out', 0)
        n_ok += 1
        msg = (f'[{num:>4}/{total_pdfs}] ✅ {pdf_path.name}'
               f' | {result["temps_total_s"]}s'
               f' | tok={result["tokens_total"]}'
               f' | {result["pages_trouvees"]}')
        if result.get('pages_manquantes'):
            msg += f' | ⚠️ {result["pages_manquantes"]}'
        if result.get('anomalies'):
            msg += f' | 🔴 {result["anomalies"]}'
        log(msg)
    except Exception as e:
        n_err += 1
        log(f'[{num:>4}/{total_pdfs}] ❌ {pdf_path.name} — {e}')
        continue

log('Génération Excel...')
rows = []
for jf in sorted(JSON_DIR.glob('*.json')):
    with open(jf, encoding='utf-8') as f:
        rows.append(json.load(f))
create_excel(EXCEL_PATH, rows)

elapsed = time.time() - t_total
log(f'✅ Terminé en {elapsed:.1f}s ({elapsed/max(1,n_ok):.1f}s/dossier)')
log(f'   Traités      : {n_ok} | Erreurs : {n_err}')
log(f'   Tokens IN    : {grand_tok_in:,}')
log(f'   Tokens OUT   : {grand_tok_out:,}')
log(f'   Tokens TOTAL : {grand_tok_in + grand_tok_out:,}')

## 13. Récupération d'urgence

In [ ]:
# Décommenter si crash kernel
# rows = []
# for jf in sorted(JSON_DIR.glob('*.json')):
#     with open(jf, encoding='utf-8') as f:
#         rows.append(json.load(f))
# create_excel(EXCEL_PATH, rows)
# print(f'✅ Excel recréé : {len(rows)} dossiers')

## 14. Analyse des résultats

In [ ]:
import pandas as pd
if EXCEL_PATH.exists():
    df = pd.read_excel(EXCEL_PATH, header=1)
    print(f'📊 {len(df)} dossiers')
    if 'Tokens total' in df.columns:
        print(f'   Tokens/dossier : {df["Tokens total"].mean():,.0f}')
    if 'Temps (s)' in df.columns:
        print(f'   Temps moyen    : {df["Temps (s)"].mean():.1f}s/dossier')
    if 'Anomalies' in df.columns:
        anom = df[df['Anomalies'].notna()]
        print(f'🔴 Doublons      : {len(anom)}')
    if 'Pages manquantes' in df.columns:
        manq = df[df['Pages manquantes'].notna()]
        print(f'⚠️  Incomplets   : {len(manq)}')
    display(df.head(5))